# 01 — Preparação dos Dados

Construção do dataset agregado por vendedor a partir das bases transacionais do Olist.

**Unidade de análise:** vendedor
**Filtros:** pedidos com status delivered, vendedores com pelo menos 5 reviews

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

DATA = "../data/"

orders = pd.read_csv(f"{DATA}olist_orders_dataset.csv",
    parse_dates=["order_purchase_timestamp", "order_delivered_customer_date",
                 "order_estimated_delivery_date"])
items = pd.read_csv(f"{DATA}olist_order_items_dataset.csv")
payments = pd.read_csv(f"{DATA}olist_order_payments_dataset.csv")
reviews = pd.read_csv(f"{DATA}olist_order_reviews_dataset.csv")
products = pd.read_csv(f"{DATA}olist_products_dataset.csv")
sellers = pd.read_csv(f"{DATA}olist_sellers_dataset.csv")
cat_trans = pd.read_csv(f"{DATA}product_category_name_translation.csv")

print(f"Orders:   {len(orders):,}")
print(f"Items:    {len(items):,}")
print(f"Payments: {len(payments):,}")
print(f"Reviews:  {len(reviews):,}")
print(f"Products: {len(products):,}")
print(f"Sellers:  {len(sellers):,}")

Orders:   99,441
Items:    112,650
Payments: 103,886
Reviews:  99,224
Products: 32,951
Sellers:  3,095


## 1. Reviews negativos por pedido

In [2]:
delivered = orders[orders["order_status"] == "delivered"].copy()
print(f"Pedidos entregues: {len(delivered):,}")

rev = reviews[["order_id", "review_score"]].merge(delivered[["order_id"]], on="order_id")
rev["is_negative"] = (rev["review_score"] <= 2).astype(int)
print(f"Reviews de pedidos entregues: {len(rev):,}")
print(f"Reviews negativos (score 1-2): {rev['is_negative'].sum():,} ({rev['is_negative'].mean()*100:.1f}%)")
print(f"\nDistribuicao review_score:")
print(rev["review_score"].value_counts().sort_index())

Pedidos entregues: 96,478
Reviews de pedidos entregues: 96,361
Reviews negativos (score 1-2): 12,347 (12.8%)

Distribuicao review_score:
review_score
1     9406
2     2941
3     7961
4    18987
5    57066
Name: count, dtype: int64


## 2. Vínculo pedido-vendedor

In [3]:
order_seller = items[["order_id", "seller_id"]].drop_duplicates("order_id")
rev = rev.merge(order_seller, on="order_id", how="left")
print(f"Reviews com seller_id: {rev['seller_id'].notna().sum():,}")

Reviews com seller_id: 96,361


## 3. Agregação por vendedor — variável resposta

In [4]:
seller_agg = rev.groupby("seller_id").agg(
    n_reviews=("review_score", "count"),
    n_negative=("is_negative", "sum"),
    review_mean=("review_score", "mean")
).reset_index()

print(f"Vendedores com reviews: {len(seller_agg):,}")
print(f"\nDistribuicao de n_reviews:")
print(seller_agg["n_reviews"].describe())

Vendedores com reviews: 2,956

Distribuicao de n_reviews:
count    2956.000000
mean       32.598444
std       104.496029
min         1.000000
25%         2.000000
50%         7.000000
75%        22.000000
max      1801.000000
Name: n_reviews, dtype: float64


## 4. Variáveis explicativas por vendedor

In [5]:
# Atraso medio de entrega
delivered["atraso_dias"] = (
    delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
).dt.days
order_atraso = delivered[["order_id", "atraso_dias"]].merge(order_seller, on="order_id")
atraso_seller = order_atraso.groupby("seller_id")["atraso_dias"].mean().reset_index(name="atraso_medio")

# Ticket medio
order_pay = payments.groupby("order_id")["payment_value"].sum().reset_index()
order_pay = order_pay.merge(order_seller, on="order_id")
ticket_seller = order_pay.groupby("seller_id")["payment_value"].mean().reset_index(name="ticket_medio")

# Frete medio
order_frete = items.groupby("order_id")["freight_value"].sum().reset_index()
order_frete = order_frete.merge(order_seller, on="order_id")
frete_seller = order_frete.groupby("seller_id")["freight_value"].mean().reset_index(name="frete_medio")

# Peso medio dos produtos
item_prod = items[["order_id", "seller_id", "product_id"]].merge(
    products[["product_id", "product_weight_g"]], on="product_id")
peso_seller = item_prod.groupby("seller_id")["product_weight_g"].mean().reset_index(name="peso_medio")

# Categoria principal do vendedor
item_cat = items[["seller_id", "product_id"]].merge(
    products[["product_id", "product_category_name"]], on="product_id")
item_cat = item_cat.merge(cat_trans, on="product_category_name", how="left")
cat_principal = item_cat.groupby("seller_id")["product_category_name_english"].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "other"
).reset_index(name="categoria_principal")

# Estado do vendedor
seller_state = sellers[["seller_id", "seller_state"]]

print("Variaveis calculadas: atraso_medio, ticket_medio, frete_medio, peso_medio, categoria_principal, seller_state")

Variaveis calculadas: atraso_medio, ticket_medio, frete_medio, peso_medio, categoria_principal, seller_state


## 5. Junção e filtros

In [6]:
df = seller_agg.merge(atraso_seller, on="seller_id", how="left")
df = df.merge(ticket_seller, on="seller_id", how="left")
df = df.merge(frete_seller, on="seller_id", how="left")
df = df.merge(peso_seller, on="seller_id", how="left")
df = df.merge(cat_principal, on="seller_id", how="left")
df = df.merge(seller_state, on="seller_id", how="left")

print(f"Antes dos filtros: {len(df):,} vendedores")
print(f"NAs por coluna:")
print(df.isnull().sum())

# Filtrar: min 5 reviews + sem NAs
df = df[df["n_reviews"] >= 5].copy()
df = df.dropna()
print(f"\nApos filtros (min 5 reviews + sem NAs): {len(df):,} vendedores")

Antes dos filtros: 2,956 vendedores
NAs por coluna:
seller_id              0
n_reviews              0
n_negative             0
review_mean            0
atraso_medio           0
ticket_medio           0
frete_medio            0
peso_medio             0
categoria_principal    0
seller_state           0
dtype: int64

Apos filtros (min 5 reviews + sem NAs): 1,754 vendedores


## 6. Resumo do dataset final

In [7]:
print(f"Shape: {df.shape}")
print(f"\nY (n_negative):")
print(f"  Media:     {df['n_negative'].mean():.2f}")
print(f"  Mediana:   {df['n_negative'].median():.0f}")
print(f"  Variancia: {df['n_negative'].var():.2f}")
print(f"  Var/Media: {df['n_negative'].var()/df['n_negative'].mean():.2f}")
print(f"  Min: {df['n_negative'].min()}, Max: {df['n_negative'].max()}")
print(f"  Zeros: {(df['n_negative']==0).sum()} ({(df['n_negative']==0).mean()*100:.1f}%)")
print(f"\nCategorias unicas: {df['categoria_principal'].nunique()}")
print(f"Estados unicos: {df['seller_state'].nunique()}")

df.describe()

Shape: (1754, 10)

Y (n_negative):
  Media:     6.88
  Mediana:   2
  Variancia: 363.66
  Var/Media: 52.90
  Min: 0, Max: 302
  Zeros: 368 (21.0%)

Categorias unicas: 65
Estados unicos: 20


,n_reviews,n_negative,review_mean,atraso_medio,ticket_medio,frete_medio,peso_medio
count,1754.000000,1754.000000,1754.000000,1754.000000,1754.000000,1754.000000,1754.000000
mean,53.575827,6.875143,4.203401,-12.112885,202.649403,25.866801,2477.636549
std,131.617196,19.069956,0.438311,4.555854,240.442604,18.492521,3693.438231
min,5.000000,0.000000,1.400000,-48.000000,20.181818,8.441000,50.000000
25%,8.000000,1.000000,4.000000,-14.333333,87.684179,17.273316,500.000000
50%,17.000000,2.000000,4.250000,-11.734604,135.852870,21.095154,1091.834565
75%,46.000000,5.000000,4.472144,-9.711905,212.724052,27.878438,2741.689935
max,1801.000000,302.000000,5.000000,17.800000,3040.112000,371.552000,30000.000000


## 7. Salvar dataset

In [8]:
df.to_csv("../data/dataset_vendedores.csv", index=False)
print(f"Salvo: data/dataset_vendedores.csv ({len(df):,} vendedores, {len(df.columns)} colunas)")

Salvo: data/dataset_vendedores.csv (1,754 vendedores, 10 colunas)
